# 새 CSV를 올려 분석하기

처음에는 [화면으로 보는 서버 사용 가이드](https://github.com/barraki72268-sketch/Koreamovenex/blob/main/docs/서버_새CSV_화면가이드.md)를 참고합니다. 서버에서는 담당자가 전달한 가이드로 확인해 주세요.

1. 왼쪽 `notebooks > 새CSV_업로드` 폴더에 CSV를 올립니다.
2. 바로 아래 설정에서 파일명, 합칠지 여부, 학습 날짜를 확인합니다.
3. 상단 `Run > Run All Cells`를 누릅니다. 결과 확인과 ZIP 다운로드까지 순서대로 표시됩니다.

회사에서 전달받은 ECT 표준 CSV용입니다. Excel 원본이나 다른 형태의 CSV는 먼저 담당자의 변환·확인이 필요합니다. 위치 1·2는 따로 분석합니다.

## 1. 여기만 설정하세요

따옴표 안의 파일명을 업로드한 이름으로 바꿉니다. `.csv.gz`는 압축을 풀지 않습니다.

| 받은 파일 | 기존 데이터와 합치기 |
|---|---|
| 이전 기간까지 포함된 전체·누적 파일 | `False` |
| 기존에 없는 새 날짜의 추가분만 있는 파일 | `True` |

아래 날짜는 기존 C10A 결과의 재현 설정입니다. 새 날짜를 학습에 넣을 때만 담당자와 정한 날짜로 바꿉니다. 학습 종료일 < 기준 설정 종료일 < 시험 날짜 순서로 정합니다.

In [ ]:
CSV_FILE = "사용법_예제.csv.gz"  # 새CSV_업로드에 올린 파일명
MERGE_EXISTING = False         # 전체 파일: False / 새 날짜 추가분: True
TRAIN_END = "2026-08-20"       # 학습에 사용할 마지막 날짜
CALIBRATION_END = "2026-10-07" # 거리 기준을 정할 마지막 날짜

## 2. 파일과 수량 확인

여기부터는 코드를 바꾸지 않습니다. 선택한 파일명과 위치별 수량이 예상과 맞는지 확인합니다. 합성 예제에는 연습용 표시가 나오며, 회사 성능으로 해석하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get("ECT_PROJECT_ROOT", str(Path.cwd())))
if not (ROOT / "ectlab").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "ectlab").is_dir():
    raise RuntimeError("프로젝트의 notebooks 폴더에서 열어 주세요.")
sys.path.insert(0, str(ROOT))
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
from IPython.display import display, HTML, FileLink
import pandas as pd
from ectlab.runtime import data_path
from ectlab.data import load_csv, inventory
from ectlab.pipeline import Settings, prepare, distance_analysis, train_models, summarize, export_run
from ectlab.plots import pca_plot, comparison_plot, true_case_plot
UPLOAD_DIR = ROOT / "notebooks" / "새CSV_업로드"
UPLOAD_DIR.mkdir(exist_ok=True)
if not isinstance(CSV_FILE, str) or Path(CSV_FILE).name != CSV_FILE or "\\" in CSV_FILE:
    raise ValueError("CSV_FILE에는 경로 없이 파일명만 입력하세요.")
if not isinstance(MERGE_EXISTING, bool):
    raise ValueError("MERGE_EXISTING은 따옴표 없이 True 또는 False로 적습니다.")
if not CSV_FILE.lower().endswith((".csv", ".csv.gz")):
    raise ValueError(".csv 또는 .csv.gz 파일을 올려 주세요. Excel은 먼저 CSV 변환이 필요합니다.")
INPUT_CSV = UPLOAD_DIR / CSV_FILE
if not INPUT_CSV.is_file():
    raise FileNotFoundError(f"새CSV_업로드 폴더에서 {CSV_FILE}을 찾지 못했습니다. 업로드 완료와 이름·확장자를 확인하세요.")
print("선택한 파일:", INPUT_CSV.name)
extra = load_csv(INPUT_CSV)
example = "사용법_예제" in INPUT_CSV.name or extra.source_file.astype(str).str.contains("합성|synthetic|demo", case=False, regex=True).any()
if example:
    display(HTML('<div style="background:#fff4d6;padding:16px;font-size:20px">합성 연습 데이터입니다. 실제 회사 성능으로 사용하지 않습니다.</div>'))
else:
    display(HTML('<div style="background:#e9f3ff;padding:16px;font-size:20px">업로드한 ECT 데이터입니다. 파일 출처와 라벨 수량을 확인하세요.</div>'))
if MERGE_EXISTING:
    if example:
        raise ValueError("사용법 예제는 회사 데이터와 합치지 않습니다. MERGE_EXISTING = False로 실행하세요.")
    base_path = data_path(ROOT, "company", os.environ.get("ECT_PRODUCT", "C10A_49513-75270R"))
    base = load_csv(base_path)
    if set(base.source_file.unique()) != set(extra.source_file.unique()):
        raise ValueError("기존 데이터와 품번·원본 파일이 다릅니다. 다른 품번은 합치지 마세요.")
    if set(base.id) & set(extra.id):
        raise ValueError("기존 데이터와 같은 측정행이 있습니다. 전체·누적 파일이면 MERGE_EXISTING = False로 실행하세요.")
    df = pd.concat([base, extra], ignore_index=True)
    print(f"기존 {len(base):,}행 + 추가 {len(extra):,}행 = {len(df):,}행")
else:
    df = extra
    print(f"업로드 파일만 분석: {len(df):,}행")
display(inventory(df))

## 3. 분석 실행

위치별 정상 기준, 거리 기법, Random Forest와 XGBoost를 계산합니다. 진행 중에는 기다려 주세요. 오류가 나오면 아래로 계속 실행하지 말고 오류 문구를 담당자에게 전달합니다.

In [ ]:
start = pd.Timestamp(TRAIN_END)
end = pd.Timestamp(CALIBRATION_END)
if start >= end:
    raise ValueError("학습 종료일은 기준 설정 종료일보다 앞서야 합니다.")
settings = Settings(train_end=start.strftime("%Y-%m-%d"), calibration_end=end.strftime("%Y-%m-%d"))
packs, counts = prepare(df, settings)
display(counts)
if not any(p["rows"].role.eq("test").any() for p in packs.values()):
    print("안내: 시험 날짜 데이터가 없습니다. 새 날짜 성능 비교는 할 수 없습니다.")
if MERGE_EXISTING:
    import datetime
    processed = ROOT / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    ANALYSIS_CSV = processed / ("업로드_병합_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".csv.gz")
    df.to_csv(ANALYSIS_CSV, index=False, encoding="utf-8-sig")
else:
    ANALYSIS_CSV = INPUT_CSV
distance_result = distance_analysis(packs)
ml_result = train_models(packs)
frames = [v for v in (distance_result["predictions"], ml_result["predictions"]) if not v.empty]
if not frames:
    raise ValueError("분석 가능한 모델 결과가 없습니다. 학습 구간 OK·마스터·라벨 수량을 확인하세요.")
predictions = pd.concat(frames, ignore_index=True)
for note in distance_result["notes"] + ml_result["notes"]:
    print("분석 안내:", note)
display(summarize(predictions))
print("분석 완료. 다음 칸에서 그래프와 저장 결과를 확인하세요.")

## 4. 그래프와 결과 저장

전체 NG 그래프는 학습에 사용한 행을 포함합니다. 시험 날짜 그래프와 구분합니다. 진성이 적거나 시험 날짜에 진성이 없으면 검출 성능을 확정할 수 없습니다.

In [ ]:
figures = {
    "01_PCA_위치별_분포": pca_plot(packs),
    "02_전체NG_가성구분": comparison_plot(predictions),
    "03_시험날짜_가성구분": comparison_plot(predictions, True),
    "04_진성행_점수": true_case_plot(predictions),
}
for title, figure in figures.items():
    display(HTML(f"<h3>{title}</h3>"))
    display(figure)
saved_folder = export_run(ROOT / "results", ANALYSIS_CSV, settings, packs, distance_result, ml_result, figures)
import shutil
zip_path = Path(shutil.make_archive(str(saved_folder), "zip", root_dir=saved_folder))
relative = saved_folder.relative_to(ROOT).as_posix()
from urllib.parse import quote
display(HTML('<h2>저장 완료</h2><p>아래 링크로 보고서를 열거나 결과 전체를 다운로드합니다.</p>'))
from ectlab.exploration import download_href
href = download_href(zip_path, ROOT)
display(HTML(f'<p><a href="{href}" download>결과 전체 ZIP 다운로드</a></p><p>압축을 풀고 결과보고서.html을 더블클릭해 확인하세요.</p>'))
print("ZIP에는 보고서, 그래프, 판정 표와 학습 모델이 들어 있습니다.")
print("결과 폴더:", saved_folder.name)

다른 CSV를 분석할 때는 새 파일을 업로드한 뒤 1번 설정을 바꾸고 `Kernel > Restart Kernel and Run All Cells…`로 다시 실행합니다. 입력 CSV는 변경하지 않으며 결과는 실행 시각별로 따로 저장합니다. 인증 링크와 회사 결과는 개별 전달 자료로 관리합니다.